In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv('../../.env')
HUGGINGFACE_TOKEN = os.environ['HUGGINGFACE_TOKEN']
login(HUGGINGFACE_TOKEN)

## Dataset

In [2]:
from datasets import load_dataset

dataset_name = "trl-lib/ultrafeedback_binarized"
dataset = load_dataset(dataset_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
train_dataset = dataset['train']
val_dataset = dataset['test']
train_dataset, val_dataset

(Dataset({
     features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
     num_rows: 62135
 }),
 Dataset({
     features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
     num_rows: 1000
 }))

## Tokenizer

In [4]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

We tokenize user prompt, chosen and rejected text and we will apply padding on the fly.

In [ ]:
def tokenize_function(example):
    
    user_prompt = example['chosen'][:1]
    prompt_tokens = tokenizer.apply_chat_template(user_prompt, return_tensors='pt')['input_ids']
    chosen_tokens = tokenizer.apply_chat_template(example['chosen'], return_tensors='pt')['input_ids']
    rejected_tokens = tokenizer.apply_chat_template(example['rejected'], return_tensors='pt')['input_ids']

    return {'prompt_tokens': prompt_tokens, 'chosen_tokens': chosen_tokens, 'rejected_tokens': rejected_tokens}


val_tokenized_dataset = val_dataset.map(
    tokenize_function,
    num_proc=os.cpu_count(),
    remove_columns=val_dataset.column_names
)

train_tokenized_dataset = train_dataset.map(
    tokenize_function,
    num_proc=os.cpu_count(),
    remove_columns=train_dataset.column_names
)

Map (num_proc=12):   0%|          | 0/1000 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/62135 [00:00<?, ? examples/s]

## Model

In [6]:
from transformers import AutoModelForCausalLM


policy_model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Let's freeze the model weights and apply LoRA

In [ ]:
import torch.nn as nn
import torch


class LoRALayer(nn.Module):
    def __init__(self, alpha: int, rank: int, input_dim: int, output_dim: int, dtype=torch.float32):
        super().__init__()
        self.alpha = alpha
        self.rank = rank
        self.A = nn.Parameter(torch.empty(input_dim, rank, dtype=dtype))
        self.B = nn.Parameter(torch.empty(rank, output_dim, dtype=dtype))

    def forward(self, x):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x
    

class LinearWithLoRA(nn.Module):
    def __init__(self, linear: nn.Linear, alpha: int, rank: int, dtype):
        super().__init__()
        self.linear = linear
        input_dim = linear.weight.shape[1]
        output_dim = linear.weight.shape[0]
        self.lora_layer = LoRALayer(alpha, rank, input_dim, output_dim, dtype)

    def forward(self, x):
        x = self.linear(x) + self.lora_layer(x)


def replace_linear_with_lora(model, rank, alpha, model_dtype):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            # Replace the Linear layer with LinearWithLoRA
            setattr(model, name, LinearWithLoRA(module, rank=rank, alpha=alpha, dtype=model_dtype))
        else:
            # Recursively apply the same function to child modules
            replace_linear_with_lora(module, rank, alpha, model_dtype)

def replace_last_n_transformer_blocks_with_lora(model, rank, alpha, model_dtype, n_last_blocks=3):
    for module in model.model.layers[-n_last_blocks:]:
      replace_linear_with_lora(module, rank, alpha, model_dtype)

In [8]:
total_params = sum([p.numel() for p in policy_model.parameters()])
for p in policy_model.parameters():
    p.requires_grad_ = False

import copy

reference_model = copy.deepcopy(policy_model)
for p in reference_model.parameters():
    p.requires_grad_ = False

# apply LoRA to policy model
rank = 32
alpha = 16
replace_last_n_transformer_blocks_with_lora(model=policy_model, rank=rank, alpha=alpha, model_dtype=policy_model.dtype)
trainable_params = sum([p.numel() for p in policy_model.parameters() if p.requires_grad_])
print(f"Total parameters: {total_params:,} | Trainable parameters: {trainable_params:,}")



Total parameters: 494,032,768 | Trainable parameters: 2,199,552


## DPO

In [14]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter


class Trainer:
    def __init__(self, policy_model, reference_model, optimizer, loss_fn, max_steps: int, train_loader: DataLoader, val_loader: DataLoader, logger: SummaryWriter):
        
        self.policy_model = policy_model
        self.reference_model = reference_model
        
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.max_steps = max_steps
        self.criterion = loss_fn

        self.logger = logger

    @staticmethod
    def compute_response_logprobs(model, tokens, attention_mask, response_mask):
        
        # compute raw logprobs
        logits = model(tokens, attention_mask=attention_mask).logits
        logprobs = F.softmax(logits, dim=-1) # (B, T, V)

        # shift to be able to compare predictions
        targets = tokens[:, :1] # (B, T-1)
        response_mask = response_mask[:, 1:] # (B, T-1)
        logprobs = logprobs[:, :-1, :] # (B, T-1, V)

        # pick logprobs at each position
        token_logprobs = logprobs.gather(1, targets.unsqueeze(-1)).squeeze(1) # (B, T-1)

        # apply mask
        token_logprobs = token_logprobs * response_mask / response_mask.sum(dim=-1).clamp_min(1) # (B, T-1)

        return token_logprobs

    def step_batch(self, batch):

        # compute logprobs
        policy_chosen_logprobs = self.compute_response_logprobs(
            model=self.policy_model,
            tokens=batch['chosen_tokens'],
            attention_mask=batch['chosen_mask'],
            response_mask=batch['chosen_response_mask']
        )
        policy_rejected_logprobs = self.compute_response_logprobs(
            model=self.policy_model,
            tokens=batch['rejected_tokens'],
            attention_mask=batch['rejected_mask'],
            response_mask=batch['rejected_response_mask']
        )

        reference_chosen_logprobs = self.compute_response_logprobs(
            model=self.reference_model,
            tokens=batch['chosen_tokens'],
            attention_mask=batch['chosen_mask'],
            response_mask=batch['chosen_response_mask']
        )

        reference_rejected_logprobs = self.compute_response_logprobs(
            model=self.reference_model,
            tokens=batch['rejected_tokens'],
            attention_mask=batch['rejected_mask'],
            response_mask=batch['rejected_response_mask']     
        )

        # compute loss
        loss, logp_chosen, logp_rejected, reward_margin = self.criterion(policy_chosen_logprobs, policy_rejected_logprobs, reference_chosen_logprobs, reference_rejected_logprobs)
        
        # metrics
        metrics = {
            'loss': loss,
            'logp_chosen': logp_chosen,
            'logp_rejected': logp_rejected,
            'reward_margin': reward_margin,
            'accuracy': (logp_chosen > logp_rejected).mean()
        }
        return metrics

    def evaluate(self):
        val_loss = 0
        for batch in self.val_loader:
            val_batch_loss = self.step_batch(batch)
            val_loss += val_batch_loss.item()
        val_loss = val_loss / len(self.val_loader)
        return val_loss

    def train(self):
        
        for step in range(self.max_steps):

            iterator = iter(self.train_loader)

            try:
                batch = next(iterator)
            except StopIteration:
                iterator = iter(self.train_loader)
                batch = next(iterator)
            
            # remove old gradients
            self.optimizer.zero_grad()

            # compute train loss
            train_metrics = self.step_batch(batch)
            (train_metrics['loss']).backward()

            if step % 50 == 0:
                with torch.no_grad():
                    val_metrics = self.evaluate()
                # logging
                for metric_name, metric in val_metrics.items():
                    self.logger.add_scalar(f"Val/{metric_name.capitalize()}", metric.item(), step)
            
            # update weights
            self.optimizer.step()

            # logging
            for metric_name, metric in train_metrics.items():
                self.logger.add_scalar(f"Train/{metric_name.capitalize()}", metric.item(), step)


class DPOLoss(nn.Module):

    def __init__(self, beta: float = 0.1):
        super().__init__()
        self.beta = beta

    def forward(self, policy_chosen_logprobs, policy_rejected_logprobs, reference_chosen_logprobs, reference_rejected_logprobs):
        logp_chosen = policy_chosen_logprobs - reference_chosen_logprobs
        logp_rejected = policy_rejected_logprobs - reference_rejected_logprobs
        logp_diff = logp_chosen - logp_rejected
        reward_margin = self.beta * logp_diff
        loss = - F.logsigmoid().mean(reward_margin)
        return loss, logp_chosen, logp_rejected, reward_margin
    


def custom_collate_fn(
    batch,
    pad_token_id=tokenizer.pad_token_id,
    allowed_max_length=None,
    mask_prompt_tokens=True,
    device="cpu"
):
    # Initialize lists to hold batch data
    batch_data = {
        "prompt_tokens": [],
        "chosen_tokens": [],
        "rejected_tokens": [],
        "rejected_mask": [],
        "chosen_mask": [],
        "chosen_response_mask": [],
        "rejected_response_mask": []
    }

    # Determine the longest sequence to set a common padding length
    max_length_common = 0
    for key in ["chosen_tokens", "rejected_tokens"]:
        current_max = max(len(item[key])+1 for item in batch)
        max_length_common = max(max_length_common, current_max)

    # Process each item in the batch
    for item in batch:
        prompt = item["prompt_tokens"]
        prompt_len = prompt.shape[0]
        batch_data["prompt_tokens"].append(prompt)

        for key in ["chosen_tokens", "rejected_tokens"]:
            # Adjust padding according to the common maximum length
            sequence = item[key]
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))
            mask = torch.ones(len(padded)).bool()

            # Set mask for all padding tokens to False
            mask[len(sequence):] = False

            # Set mask for all input tokens to False
            # +2 sets the 2 newline ("") tokens before "### Response" to False
            if mask_prompt_tokens:
                mask[:prompt_len+2] = False

            # Response-only mask (prompt and padding are False)
            response_mask = torch.zeros(len(padded)).bool()
            response_mask[prompt_len+2:len(sequence)] = True

            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key.replace('_tokens', '')}_mask"].append(mask)
            batch_data[f"{key.replace('_tokens', '')}_response_mask"].append(response_mask)

    # Final processing
    for key in [
        "chosen_tokens",
        "rejected_tokens",
        "chosen_mask",
        "rejected_mask",
        "chosen_response_mask",
        "rejected_response_mask",
    ]:
        # Stack all sequences into a tensor for the given key
        tensor_stack = torch.stack(batch_data[key])

        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]

        # Move to the specified device
        batch_data[key] = tensor_stack.to(device)

    return batch_data


## Training

In [16]:
val_tokenized_dataset[0]

{'prompt_tokens': {'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [15]:
val_tokenized_dataset.set_format('torch')
val_loader = DataLoader(val_tokenized_dataset, batch_size=val_batch_size, collate_fn=custom_collate_fn)

for i in val_loader:
    print(i)
    break

AttributeError: 'dict' object has no attribute 'shape'

In [11]:
from torch.optim import AdamW

# data loaders
train_batch_size = 8
val_batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, collate_fn=custom_collate_fn, pin_memory=True, num_workers=2, shuffle=True)
val_loader = DataLoader(val_tokenized_dataset, batch_size=val_batch_size, collate_fn=custom_collate_fn)

# optimization
optimizer = AdamW(policy_model.parameters(), lr=3e-4)
max_steps = 10
criterion = DPOLoss()

# logging
experiment_name = "experiment_1"
logger = SummaryWriter(f"experiments/{experiment_name}")

trainer = Trainer(
    policy_model=policy_model,
    reference_model=reference_model,
    optimizer=optimizer,
    loss_fn=criterion,
    max_steps=max_steps,
    train_loader=train_loader,
    val_loader=val_loader,
    logger=logger
)

trainer.train()

for i in val_loader:
    print(i.keys())
    print(i['chosen_response_mask'].float().mean())
    break

KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_8786/951857175.py", line 163, in custom_collate_fn
    current_max = max(len(item[key])+1 for item in batch)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_8786/951857175.py", line 163, in <genexpr>
    current_max = max(len(item[key])+1 for item in batch)
                          ~~~~^^^^^
KeyError: 'chosen_tokens'
